In [ ]:
pip install trl datasets transformers peft torch

In [2]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

In [3]:
# ──────────────────────────────────────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────────────────────────────────────
# Gemma 3 270M-it : meilleur suivi d'instructions que SmolLM2-135M
# (51.2% vs 38% sur IFEval), tourne sur CPU en quelques minutes avec 5 exemples.
# Nécessite d'accepter les CGU sur https://huggingface.co/google/gemma-3-270m-it
# Ou alternative sans inscription : "facebook/opt-125m"


# MODEL_NAME = "google/gemma-3-270m-it"

# Sinon ce modèle, pas besoin d'accepter les conditions de Google, et il marche bien
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
OUTPUT_DIR = "./it_assistant_finetuned"
USE_CPU_ONLY = False #A mettre sur True si vous n'avez pas de GPU (garder sur False sur GOogle Colab)

# Préparation des données

In [51]:
# Exemples métier à reconstruire à partir des sources de données disponibles
# Ceci est un exemple vous devez reconstruire une liste Python similaire
EXEMPLES_METIER = [
  {
    "instruction": "Comment réinitialiser mon mot de passe Windows ?",
    "reponse": "Pour réinitialiser ton mot de passe Windows :\n1. Rends-toi sur AtosConnect > Sécurité > Réinitialisation.\n2. Saisis ton identifiant employé (format ATO-XXXXX).\n3. Un lien temporaire valable 15 minutes t'est envoyé par SMS.\n4. Choisis un mot de passe d'au moins 12 caractères.\nEn cas de blocage, contacte le support au poste 4242."
  },
  {
    "instruction": "Mon VPN ne se connecte plus depuis ce matin.",
    "reponse": "Pour résoudre le problème de VPN :\n1. Vérifie que tu es bien connecté à Internet hors VPN.\n2. Redémarre le client GlobalProtect (icône dans la barre système).\n3. Si l'erreur persiste, vide le cache : AtosConnect > VPN > Réinitialiser.\n4. Redémarre ton poste et relance GlobalProtect.\nSi le problème persiste après ces étapes, ouvre un ticket P2 sur AtosConnect."
  },
]

In [52]:
#Formatage en prompt d'instruction

PROMPT_TEMPLATE = """### Instruction:
{instruction}

### Réponse:
{reponse}"""

In [53]:
# La fonction qui va formatter en entrée du modèle

def formatter(example):
    return {"text": PROMPT_TEMPLATE.format(
        instruction=example["instruction"],
        reponse=example["reponse"],
    )}

In [ ]:
dataset = Dataset.from_list(EXEMPLES_METIER).map(formatter)

In [ ]:
print(f"Dataset : {len(dataset)} exemples")
print("\nExemple de prompt formaté :")
print(dataset[0]["text"])
print("─" * 60)

# Model à finetuner

In [55]:
# Chargement du modèle et du tokenizer

print(f"\nChargement du modèle : {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


Chargement du modèle : HuggingFaceTB/SmolLM2-135M


In [56]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="auto",
)
print(f"Paramètres : {model.num_parameters():,}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Paramètres : 134,515,008


In [72]:
#Fonction d'inférence réutilisable (appelée avant ET après fine-tuning)

# Observez notamment la variables inputs, nous avons besoind e tokenizer avant de passer en entrée du modèle

def generer_reponse(model, tokenizer, question, max_new_tokens=256):
    """Génère une réponse à partir d'une question, sans contexte."""
    prompt = (
        PROMPT_TEMPLATE.format(instruction=question, reponse="")
        .rstrip()
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Inférence AVANT fine-tuning

In [58]:
# ──────────────────────────────────────────────────────────────────────────────
#   Inférence AVANT fine-tuning
#    On pose la même question que dans les exemples d'entraînement.
#    Le modèle de base va répondre de façon générique, sans connaître
#    AtosConnect, GlobalProtect, ni le format interne.
# ──────────────────────────────────────────────────────────────────────────────
QUESTION_TEST = "Comment réinitialiser mon mot de passe Windows ?"

In [59]:
print("\n" + "═" * 60)
print("INFÉRENCE AVANT FINE-TUNING")
print("═" * 60)
reponse_avant = generer_reponse(model, tokenizer, QUESTION_TEST)
print(f"Question : {QUESTION_TEST}")
print(f"\nRéponse  :\n{reponse_avant}")


════════════════════════════════════════════════════════════
INFÉRENCE AVANT FINE-TUNING
════════════════════════════════════════════════════════════
Question : Comment réinitialiser mon mot de passe Windows ?

Réponse  :
Comment réinitialiser mon mot de passe Windows ?

### Réponse:
Comment réinitialiser mon mot de passe Windows ?

### Réponse:
Comment réinitialiser mon mot de passe Windows ?

### Réponse:
Comment réinitialiser mon mot de passe Windows ?

### Réponse:
Comment réinitialiser mon mot de passe Windows ?

### Réponse:
Comment réinitialiser mon mot de passe Windows ?

### Réponse:
Comment réinitialiser mon mot de passe Windows ?

### Réponse:
Comment r


# Fine-tuning LoRA

In [76]:
# ──────────────────────────────────────────────────────────────────────────────
#  Fine-tuning LoRA sur les exemples
#
#    Hyperparamètres volontairement agressifs (learning_rate élevé,
#    num_train_epochs élevé) pour forcer l'apprentissage sur un si
#    petit dataset. En production avec un vrai dataset (>500 exemples),
#    on utiliserait des valeurs plus conservatrices.
# ──────────────────────────────────────────────────────────────────────────────

In [61]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,           # rang plus élevé pour compenser le très petit dataset
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

In [64]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=40,          # epochs élevés
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,           # lr plus agressif pour petit dataset
    max_length=256,
    dataset_text_field="text",
    push_to_hub=False, # On ne veut pas pousser sur le hub de huggingface !
    report_to="none",
    logging_steps=5,              # affiche la loss toutes les 5 steps
    save_strategy="no",           # pas de checkpoint intermédiaire
    use_cpu =USE_CPU_ONLY
)

In [20]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [65]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

In [66]:
trainer.train()
print("Fine-tuning terminé.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
5,3.063377
10,2.880935
15,2.864580
20,2.792182
25,2.672728
30,2.575310
35,2.393370
40,2.555953
45,2.407854
50,2.330643


Fine-tuning terminé.


# Inférence après finetuning

In [75]:
# ──────────────────────────────────────────────────────────────────────────────
# Inférence APRÈS fine-tuning — même question, même modèle
# ──────────────────────────────────────────────────────────────────────────────
model.eval()
reponse_apres = generer_reponse(model, tokenizer, QUESTION_TEST)
print(f"Question : {QUESTION_TEST}")
print(f"\nRéponse  :\n{reponse_apres}")

Question : Comment réinitialiser mon mot de passe Windows ?

Réponse  :
Pour réinitialiser ton mot de passe Windows :
1. Vérifie que tu espere vers ton mot de passe Windows (voir tandis page 2).
2. Ouvre une demande de réinitialisation sur AtosConnect > Sécurité > Réinitialisation.
3. Si ton mot de passe est définir le mot de passe de réinitialisation.
4. Si un demande de réinitialisation n'est définir, ouvre un ticket P2 sur AtosConnect > Sécurité > Réinitialisation avant l'é Vernir.


In [74]:
# ──────────────────────────────────────────────────────────────────────────────
# 9. Test sur une question proche mais PAS dans le dataset d'entraînement
#    → vérifie que le modèle généralise le format, pas qu'il mémorise bêtement
# ──────────────────────────────────────────────────────────────────────────────
QUESTION_HORS_DATASET = "Mon imprimante ne répond plus, comment la reconnecter ?"

print(f"\nTest de généralisation (question hors dataset) :")
print(f"Question : {QUESTION_HORS_DATASET}")
reponse_hors = generer_reponse(model, tokenizer, QUESTION_HORS_DATASET)
print(f"\nRéponse  :\n{reponse_hors}")
print("\n→ Si le modèle utilise le format en étapes et mentionne AtosConnect,")
print("  c'est qu'il a bien généralisé le style, pas juste mémorisé.")


Test de généralisation (question hors dataset) :
Question : Mon imprimante ne répond plus, comment la reconnecter ?

Réponse  :
Pour résoudre un imprimante qui ne répond plus :
1. Vérifie que le corrélateur n'est pas encore : sékwiesi, sérialisateur, ou bien le meme poste.
2. Va dans Paramètres > Catalogue de services > Imprimante.
3. Déconnecte-toi le corépument et reconnecte-toi le catalogue réseau Atos (voir Paramètres réseau > Corépument).
4. Redéconnecte-toi le corépument et reconnecte-toi le catalogue réseau Atos.
Si l'impression est déjà correct, ouvre un ticket sur AtosConnect > Méthode > Profil de l'imprimante.

→ Si le modèle utilise le format en étapes et mentionne AtosConnect,
  c'est qu'il a bien généralisé le style, pas juste mémorisé.
